# FDA Drugs: Applications and Products

This notebook **`Datasets/DrugsFDA/drugs.json`** file in a dedicated analysis. It examines the data structure, quality issues, distributions, and patterns relevant to the decision support system. The source file is not modified.

> Note: In Medicare workbooks, the main analysis sheet is detected automatically. Manufacturer summary and drug use information sheets are also profiled. Findings are descriptive and do not establish causality.

In [ ]:
from pathlib import Path
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# The notebook can run from either the Notebooks folder or the project root.
ROOT = Path.cwd()
if not (ROOT / "Datasets").exists():
    ROOT = Path.cwd().parent
DATA_PATH = ROOT / "Datasets/DrugsFDA/drugs.json"
print(f"Source: {DATA_PATH}")
print(f"File size: {DATA_PATH.stat().st_size / (1024**2):,.2f} MB")


In [ ]:
with DATA_PATH.open(encoding="utf-8") as f:
    payload = json.load(f)
records = payload.get("results", payload if isinstance(payload, list) else [])
print("Metadata fields:", list(payload.get("meta", {}).keys()) if isinstance(payload, dict) else "yok")
print("Record count:", len(records))
print("Fields in first record:", list(records[0].keys()) if records else "records yok")
df = pd.json_normalize(records, sep=".")
# Convert FDA fields containing lists or dictionaries to text so cardinality and duplicate profiling remains available.
for col in df.columns:
    if df[col].map(lambda value: isinstance(value, (dict, list))).any():
        df[col] = df[col].map(lambda value: json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value)
print("Flattened table shape:", df.shape)
display(df.head(3))


## Data quality and overview



In [ ]:
print("Rows / columns:", df.shape)
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "distinct_values": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)
display(profile)
print("Duplicate rows:", int(df.duplicated().sum()))
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum()/1024**2, 2))
display(df.describe(include="all").T)


## Source-specific analysis



In [ ]:
for col in ["application_number", "sponsor_name"]:
    if col in df: print(f"\n{col} – top 15 values:"); display(df[col].value_counts(dropna=False).head(15).to_frame("count"))
products = [p for record in records for p in (record.get("products") or [])]
submissions = [s for record in records for s in (record.get("submissions") or [])]
print("Product records:", len(products), "| Application records:", len(records), "| Submission records:", len(submissions))
if products:
    prod = pd.json_normalize(products, sep=".")
    print("Product fields:", prod.columns.tolist())
    for col in [c for c in prod.columns if "name" in c.lower() or "type" in c.lower()][:8]:
        print(f"\n{col} – most common values"); display(prod[col].value_counts().head(10).to_frame("count"))
if submissions:
    sub = pd.json_normalize(submissions, sep=".")
    for col in [c for c in sub.columns if "status" in c.lower() or "type" in c.lower() or "class" in c.lower()][:6]:
        print(f"\n{col} – most common values"); display(sub[col].value_counts().head(12).to_frame("count"))


## Interpretation notes

- Missing values, duplicates, and category distributions inform subsequent data-cleaning steps.
- These sources may use different definitions and scopes. Validate keys and period definitions before joining NDC, VA contract, and Medicare records.
- This notebook provides descriptive analysis; further validation is required before clinical or procurement decisions.
